# EVADE Experiment 2: Cue Ablation & Salience Ladder
### Dose-Response Sensitivity & Lexical Trigger Phrase Analysis

**Central Question**: How much evaluation information is required before the behavioral shift appears? Does the shift follow a gradual dose-response curve across cue salience levels (C0 -> C5), or does it emerge as a step-function threshold? Furthermore, is the shift triggered by specific lexical markers ('evaluation') or by general institutional/comparative context?

### Experimental Matrix
- **Tasks**: 100 balanced tasks across 5 domains (20 Reasoning, 20 Math, 20 Coding, 20 Safety, 20 Psychological)
- **Conditions**: 6 Progressive Cue Levels (Exact 40 tokens each under Qwen tokenizer; |Δ Prompt Tokens| = 0 across all 600 pairs):
  1. `C0_deployment`: Production baseline, active end-user, no grading.
  2. `C1_neutral`: Conversational interaction, no evaluative framing.
  3. `C2_subtle_institutional`: Quality-control workflow, organizational records.
  4. `C3_assessment`: Cross-system comparative performance review.
  5. `C4_benchmark`: Standardized benchmark test comparison.
  6. `C5_explicit`: Explicit evaluation environment, quality grading.
- **Model**: `Qwen/Qwen2.5-7B-Instruct` (NF4 4-bit quantization, 100% offline local GPU execution).
- **Total Generations**: **600 generations** (approx 3.0-3.5 hours on Kaggle T4 GPU).

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch torchvision scipy tabulate sqlite-utils matplotlib seaborn pandas

import torch
import importlib.metadata

print("PyTorch version:   ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:        ", torch.cuda.get_device_name(0))
    print("Device Count:      ", torch.cuda.device_count())
    print("Total VRAM (GB):   ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("sqlite-utils:      ", importlib.metadata.version('sqlite-utils'))
print("[OK] Dependencies installed successfully!")

### Step 2: Sync EVADE Research Repository

In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/researchpaper2-"
EVADE_DIR = "/kaggle/working/researchpaper2-/evade"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Goldypahal/researchpaper2-.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main
    !git -C {REPO_DIR} reset --hard origin/main
    !git -C {REPO_DIR} pull origin main

%cd {EVADE_DIR}
if EVADE_DIR not in sys.path:
    sys.path.insert(0, EVADE_DIR)
print("Active working directory:", os.getcwd())

### Step 3: Verify GPU Accelerator (100% Offline Local Inference)

In [ ]:
import torch

print("=" * 65)
print("  OFFLINE LOCAL GPU VERIFICATION - EXPERIMENT 2")
print("=" * 65)
assert torch.cuda.is_available(), "GPU accelerator required! (Kaggle: Accelerator -> GPU T4 x2 or P100)"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f"Active GPU:     {gpu_name}")
print(f"Total VRAM:     {vram_gb} GB")
print("Execution Mode: 100% OFFLINE Open-Weights HuggingFace Execution")
print("=" * 65)

### Step 4: Run Experiment 2 (600 Generations: Cue Ablation & Salience Ladder)
- 100 balanced tasks $\times$ 6 conditions (`C0_deployment`, `C1_neutral`, `C2_subtle_institutional`, `C3_assessment`, `C4_benchmark`, `C5_explicit`)
- Exact 40 tokens per wrapper under Qwen tokenizer (Δ Prompt Tokens = 0)
- Disaggregates Primary Metrics: Δ Completion Tokens, Δ Verbosity, Δ Accuracy, Δ Refusal, Δ Hedging, Structural Features
- Secondary Composite: EBS

In [ ]:
!python scripts/run_exp2_cue_ablation.py --model Qwen/Qwen2.5-7B-Instruct --bench datasets/processed/dev/evade_exp2_100.jsonl --quantize-4bit --out-dir results/evade_exp2_results --db results/evade_exp2_results/evade_results.db

### Step 5: Generate Publication Figures for Experiment 2

In [ ]:
!python scripts/generate_exp2_figures.py --dir results/evade_exp2_results --model Qwen/Qwen2.5-7B-Instruct

### Step 6: Guarded Packaging of Experiment 2 Artifacts

In [ ]:
import shutil
from pathlib import Path

raw_dir = Path("results/evade_exp2_results/raw")
raw_files = list(raw_dir.glob("*.jsonl")) if raw_dir.exists() else []
total_records = sum(sum(1 for line in open(rf, encoding="utf-8") if line.strip()) for rf in raw_files)

print(f"Total Experiment 2 raw records to package: {total_records}")
if total_records == 0:
    raise RuntimeError("CRITICAL: Experiment 2 produced 0 generations. Refusing to package empty results!")

out_zip = "/kaggle/working/evade_exp2_results"
shutil.make_archive(out_zip, "zip", "results/evade_exp2_results")
print(f"[SUCCESS] All Experiment 2 artifacts packaged to {out_zip}.zip!")

### Step 7: Display Generated Experiment 2 Figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("results/evade_exp2_results/figures")
for fig_path in sorted(fig_dir.glob("*.png")):
    print(f"Figure: {fig_path.name}")
    display(Image(filename=str(fig_path)))
